# Caso 07 — Ampara, seguros: ¿qué canal paga lo que cuesta?
## Día 4 — ¿Deja margen o cuesta más de lo que paga?

Hoy cerramos el análisis en SQL uniendo las cuatro piezas que definen la rentabilidad real de cada canal en Ampara:
1. **Combined Ratio por canal**:
   $$\text{Combined Ratio} = \text{Loss Ratio} + 22% \text{ (gastos fijos)} + \frac{\text{CAC}}{\text{Prima media}}$$
   Verificamos qué canales superan la línea crítica del **100%** (donde cada póliza pierde dinero en suscripción).
2. **Tasa de Renovación (`SELF JOIN` sobre `polizas`)**:
   Una póliza nueva (`tipo = 'nueva'`) se ha renovado si su `poliza_id` aparece como `poliza_origen_id` de alguna póliza de renovación.
3. **Margen Técnico por Póliza y Margen Técnico Total por Canal**:
   $$\text{Margen por póliza} = \text{Prima media} \times (1 - \text{Combined Ratio})$$
   $$\text{Margen total} = \text{Margen por póliza} \times N_{\text{pólizas nuevas}}$$
   Comprobamos cuánto margen genera Ampara en total (`+736.320 €`) y cuánto generaría sin el agujero de los comparadores (`+1.557.120 €`, más del doble).
4. **Control de Confusión (Descartar la corazonada de Lucía):**
   Calculamos el **Loss Ratio por canal dentro de cada uno de los 5 ramos** (`auto`, `salud`, `mascotas`, `hogar`, `vida`) para demostrar que la brecha de ~30 puntos porcentuales entre `comparadores` y `referido` se mantiene en todos los ramos: no es el auto, es la **selección adversa** del canal.
5. **El Contrapunto de Mediadores:** por qué el canal con el CAC más alto (`90 €`) es el segundo más rentable (`+56 €/póliza`, `+609.120 €` totales).


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

con = duckdb.connect()
for tabla in ["canales", "productos", "mediadores", "campanas", "clientes", "polizas", "siniestros"]:
    con.execute(f"CREATE OR REPLACE VIEW {tabla} AS SELECT * FROM read_csv_auto('data/clean/{tabla}.csv')")


### 1. Tabla Maestra en SQL: Combined Ratio, Renovación (`SELF JOIN`), Margen por Póliza y Margen Total
Unimos en una sola consulta con CTEs las cuatro piezas (siniestros, gastos del 22%, captación y renovación).


In [2]:
query_maestra_d4 = """
WITH polizas_nuevas AS (
    SELECT poliza_id, canal, producto_id, prima_anual
    FROM polizas
    WHERE tipo = 'nueva' AND prima_anual > 0
),
renovaciones_unicas AS (
    SELECT DISTINCT CAST(poliza_origen_id AS BIGINT) AS poliza_origen_id
    FROM polizas
    WHERE poliza_origen_id IS NOT NULL
),
metricas_polizas AS (
    SELECT
        p.canal,
        COUNT(*) AS polizas_nuevas,
        SUM(p.prima_anual) AS prima_total_eur,
        AVG(p.prima_anual) AS prima_media_eur,
        COUNT(r.poliza_origen_id) AS polizas_renovadas,
        1.0 * COUNT(r.poliza_origen_id) / COUNT(*) AS tasa_renovacion
    FROM polizas_nuevas p
    LEFT JOIN renovaciones_unicas r ON p.poliza_id = r.poliza_origen_id
    GROUP BY p.canal
),
metricas_siniestros AS (
    SELECT
        p.canal,
        SUM(s.importe) AS siniestros_total_eur
    FROM polizas_nuevas p
    LEFT JOIN siniestros s ON p.poliza_id = s.poliza_id
    GROUP BY p.canal
),
metricas_campanas AS (
    SELECT
        canal,
        SUM(coste) AS coste_campanas_eur
    FROM campanas
    GROUP BY canal
),
consolidado AS (
    SELECT
        mp.canal,
        mp.polizas_nuevas,
        ROUND(100.0 * mp.polizas_nuevas / SUM(mp.polizas_nuevas) OVER (), 2) AS cuota_polizas_pct,
        ROUND(mp.prima_total_eur, 2) AS prima_total_eur,
        ROUND(100.0 * mp.prima_total_eur / SUM(mp.prima_total_eur) OVER (), 2) AS cuota_prima_pct,
        ROUND(mp.prima_media_eur, 2) AS prima_media_eur,
        ROUND(mc.coste_campanas_eur / mp.polizas_nuevas, 2) AS cac_eur,
        ms.siniestros_total_eur / mp.prima_total_eur AS loss_ratio,
        0.22 AS gastos_ratio,
        (mc.coste_campanas_eur / mp.polizas_nuevas) / mp.prima_media_eur AS cac_ratio,
        (ms.siniestros_total_eur / mp.prima_total_eur) + 0.22 + ((mc.coste_campanas_eur / mp.polizas_nuevas) / mp.prima_media_eur) AS combined_ratio,
        mp.tasa_renovacion
    FROM metricas_polizas mp
    LEFT JOIN metricas_siniestros ms ON mp.canal = ms.canal
    LEFT JOIN metricas_campanas mc ON mp.canal = mc.canal
)
SELECT
    canal,
    polizas_nuevas,
    cuota_polizas_pct,
    prima_total_eur,
    cuota_prima_pct,
    prima_media_eur,
    cac_eur,
    ROUND(100.0 * loss_ratio, 2) AS loss_ratio_pct,
    ROUND(100.0 * gastos_ratio, 2) AS gastos_pct,
    ROUND(100.0 * cac_ratio, 2) AS cac_sobre_prima_pct,
    ROUND(100.0 * combined_ratio, 2) AS combined_ratio_pct,
    ROUND(100.0 * tasa_renovacion, 2) AS renovacion_pct,
    ROUND(prima_media_eur * (1.0 - combined_ratio), 2) AS margen_por_poliza_eur,
    ROUND(polizas_nuevas * prima_media_eur * (1.0 - combined_ratio), 2) AS margen_total_eur
FROM consolidado
ORDER BY margen_total_eur ASC
"""
df_maestra = con.execute(query_maestra_d4).df()
display(df_maestra)

margen_total_ampara = df_maestra["margen_total_eur"].sum()
margen_sin_comparadores = df_maestra[df_maestra["canal"] != "comparadores"]["margen_total_eur"].sum()
print(f"\n>>> Margen Técnico Total de Ampara (5 canales): {margen_total_ampara:,.2f} €")
print(f">>> Margen Técnico de Ampara SIN comparadores:  {margen_sin_comparadores:,.2f} € ({margen_sin_comparadores / margen_total_ampara:.2f}x el margen actual)")


,canal,polizas_nuevas,cuota_polizas_pct,prima_total_eur,cuota_prima_pct,prima_media_eur,cac_eur,loss_ratio_pct,gastos_pct,cac_sobre_prima_pct,combined_ratio_pct,renovacion_pct,margen_por_poliza_eur,margen_total_eur
0,comparadores,24000,40.0,11520000.07,40.25,480.0,15.0,82.0,22.0,3.12,107.13,41.0,-34.2,-820800.44
1,directo,13200,22.0,5543999.73,19.37,420.0,30.0,68.0,22.0,7.14,97.14,62.0,12.0,158400.13
2,banca,7200,12.0,2807999.68,9.81,390.0,45.0,58.0,22.0,11.54,91.54,71.0,33.0,237599.97
3,referido,4800,8.0,2159999.95,7.55,450.0,20.0,48.0,22.0,4.44,74.44,80.0,115.0,551999.81
4,mediadores,10800,18.0,6587999.95,23.02,610.0,90.0,54.0,22.0,14.75,90.75,78.0,56.4,609120.11



>>> Margen Técnico Total de Ampara (5 canales): 736,319.58 €
>>> Margen Técnico de Ampara SIN comparadores:  1,557,120.02 € (2.11x el margen actual)


### 2. Descartar la Corazonada de Lucía: Control de Confusión (Loss Ratio por Canal dentro de cada Ramo)
Lucía Vega (Head de Suscripción) planteó una hipótesis alternativa razonable: *«¿Y si los comparadores salen mal solo porque el 60% de lo que venden es auto, que es el ramo más siniestral?»*.
Calculamos en SQL el **loss ratio por canal dentro de cada ramo** para comprobar si la brecha desaparece o se mantiene.


In [3]:
query_control_ramo = """
WITH polizas_ramo AS (
    SELECT
        p.poliza_id,
        p.canal,
        pr.ramo,
        p.prima_anual
    FROM polizas p
    JOIN productos pr ON p.producto_id = pr.producto_id
    WHERE p.tipo = 'nueva' AND p.prima_anual > 0
),
prima_canal_ramo AS (
    SELECT
        ramo,
        canal,
        COUNT(*) AS polizas_nuevas,
        SUM(prima_anual) AS prima_total_eur
    FROM polizas_ramo
    GROUP BY ramo, canal
),
siniestros_canal_ramo AS (
    SELECT
        pr.ramo,
        pr.canal,
        SUM(s.importe) AS siniestros_total_eur
    FROM polizas_ramo pr
    LEFT JOIN siniestros s ON pr.poliza_id = s.poliza_id
    GROUP BY pr.ramo, pr.canal
)
SELECT
    pcr.ramo,
    pcr.canal,
    pcr.polizas_nuevas,
    ROUND(pcr.prima_total_eur, 2) AS prima_total_eur,
    ROUND(scr.siniestros_total_eur, 2) AS siniestros_total_eur,
    ROUND(100.0 * scr.siniestros_total_eur / pcr.prima_total_eur, 2) AS loss_ratio_pct
FROM prima_canal_ramo pcr
LEFT JOIN siniestros_canal_ramo scr ON pcr.ramo = scr.ramo AND pcr.canal = scr.canal
ORDER BY pcr.ramo, loss_ratio_pct DESC
"""
df_control_ramo = con.execute(query_control_ramo).df()
piv_loss_ramo = df_control_ramo.pivot(index="ramo", columns="canal", values="loss_ratio_pct")[
    ["comparadores", "directo", "banca", "mediadores", "referido"]
]
piv_loss_ramo["brecha_comp_vs_ref_pp"] = (piv_loss_ramo["comparadores"] - piv_loss_ramo["referido"]).round(2)
piv_loss_ramo = piv_loss_ramo.sort_values("comparadores", ascending=False)
print("=== CONTROL DE CONFUSIÓN: LOSS RATIO (%) POR CANAL DENTRO DE CADA RAMO ===")
display(piv_loss_ramo)


=== CONTROL DE CONFUSIÓN: LOSS RATIO (%) POR CANAL DENTRO DE CADA RAMO ===


canal,comparadores,directo,banca,mediadores,referido,brecha_comp_vs_ref_pp
ramo,,,,,,
auto,89.71,79.38,70.80,67.49,58.76,30.95
salud,73.99,63.79,56.83,55.88,47.17,26.82
mascotas,68.56,56.78,53.39,49.45,41.08,27.48
hogar,58.55,49.11,45.89,42.07,35.41,23.14
vida,53.57,44.05,43.36,42.43,30.50,23.07


### 3. Síntesis del Día 4: Una Sola Raíz Causal (La Selección Adversa) y el Contrapunto de Mediadores
- **Una sola cadena causal**: el comparador ordena por precio y atrae al *cazador de precio* (selección adversa) $\rightarrow$ siniestra mucho más (**Loss Ratio 82,0%**) $\rightarrow$ supera el umbral de rentabilidad en suscripción (**Combined Ratio 107,1%**) $\rightarrow$ se marcha al año siguiente cuando encuentra otra oferta barata (**Renovación 41,0%**) $\rightarrow$ destruye **`-34,20 €` por póliza** y **`-820.800 €` en total**.
- **El contrapunto de `mediadores`**: tiene el CAC más alto de todos (**`90 €`**, seis veces el de comparadores), pero gracias a su baja siniestralidad (**Loss Ratio 54,0%**) y alta fidelidad (**Renovación 78,0%**), logra un **Combined Ratio del 90,8%** y deja **`+56,40 €/póliza`** (**`+609.120 €` totales**, el canal que más euros absolutos aporta al margen de Ampara). Demuestra que **«CAC bajo = bueno» es un espejismo**.
